In [1]:
import re
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# Step 1: Load content from GeeksforGeeks Aurora article
url = "https://www.geeksforgeeks.org/devops/amazon-aurora/"
loader = WebBaseLoader(url)
contents = loader.load()

# Step 2: Clean and chunk text
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

text = "\n".join([doc.page_content for doc in contents])
cleaned = re.sub(r'\n+', '\n', text)

chunks = splitter.split_text(cleaned)
documents = [Document(page_content=chunk) for chunk in chunks]

print(f"Created {len(documents)} chunks")

# Step 3: Initialize embeddings
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# Step 4: Create FAISS vectorstore
db = FAISS.from_documents(documents, embedding)

# Step 5: Build RetrievalQA pipeline
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff"
)

# Step 6: Query the system
query = "Explain Amazon Aurora architecture and its advantages."
answer = qa.invoke(query)

print("===== Query =====")
print(query)
print("===== Answer =====")
print(answer)


Created 24 chunks


D:\Ethans\Python\AI-Automation\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


===== Query =====
Explain Amazon Aurora architecture and its advantages.
===== Answer =====
Amazon Aurora features a cloud-native architecture that separates compute from storage, which is fundamentally different from traditional databases. Here are some key aspects of its architecture and advantages:

1. **Log-Structured Distributed Storage System**: Unlike traditional databases that write data to a local disk (EBS volume), Aurora uses a log-structured distributed storage system. This allows for more efficient data management and retrieval.

2. **Data Replication**: Aurora maintains six copies of data across three Availability Zones (AZs), ensuring high availability and durability. This replication helps in quick recovery and minimizes data loss.

3. **Performance**: Aurora provides up to 5 times the throughput of standard MySQL and 3 times that of standard PostgreSQL, making it significantly faster than traditional RDS options.

4. **Auto-Scaling Storage**: Aurora can automatically s